# VisionTalk — Image Captioning with VGG16 + LSTM

VisionTalk is an image-captioning project built on the **Flickr8k dataset**. A pre-trained VGG16 network extracts visual features and an LSTM decoder generates a caption word by word.

## Architecture
- **Image encoder:** VGG16 `fc2` 4096-dimensional feature vector.
- **Text decoder:** Embedding + LSTM.
- **Fusion:** Keras Functional API `Add` merge.
- **Output:** Dense layer followed by Softmax for next-word prediction.
- **Inference:** greedy next-word decoding.

> **Final-run notebook:** The expensive dataset preparation, feature extraction and model training were already completed for this project. The cells below inspect and present the generated artifacts and final evaluation results; they do not retrain the model.


## 1. Project structure

Expected project layout:

```text
VisionTalk/
├── app/
│   └── streamlit_app.py
├── data/
│   ├── Images/
│   └── captions.txt
├── models/
│   ├── features.pkl
│   ├── tokenizer.pkl
│   └── visiontalk_model.keras
├── outputs/
│   ├── bleu_scores.json
│   ├── metrics.txt
│   ├── predictions_test.csv
│   ├── sample_predictions.png
│   └── evaluation/
│       └── 10_test_images.png
└── src/
```


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
MODELS = ROOT / "models"
OUTPUTS = ROOT / "outputs"
EVALUATION = OUTPUTS / "evaluation"

required = [
    MODELS / "visiontalk_model.keras",
    MODELS / "tokenizer.pkl",
    MODELS / "features.pkl",
    OUTPUTS / "bleu_scores.json",
    OUTPUTS / "predictions_test.csv",
    OUTPUTS / "metrics.txt",
    EVALUATION / "10_test_images.png",
]

for path in required:
    print(("OK   " if path.exists() else "MISS ") + str(path))


## 2. Dataset

The project uses **Flickr8k** with images under `data/Images/` and the caption file at `data/captions.txt`. The completed evaluation used **809 test images**.

The trained artifacts were generated from the Flickr8k dataset; no evaluation score is hard-coded into the model.


## 3. Model pipeline

### Image branch
Flickr8k image → resize to 224×224 → VGG16 → `fc2` 4096-D visual feature.

### Text branch
Caption prefix → tokenization/padding → Embedding → LSTM.

### Merge and prediction
The image and text representations are merged with an element-wise `Add`, followed by Dense layers and Softmax to predict the next word.


## 4. Training configuration used by the project

| Setting | Value |
|---|---:|
| Backbone | VGG16 |
| Image size | 224 × 224 |
| Feature dimension | 4096 |
| Embedding dimension | 256 |
| LSTM units | 256 |
| Dense units | 256 |
| Dropout | 0.5 |
| Batch size | 32 |
| Epochs | 20 |
| Train split | 85% |
| Random seed | 42 |


## 5. Final evaluation results

These values are the results from the completed evaluation run on the held-out test set.


In [ ]:
scores_path = OUTPUTS / "bleu_scores.json"
with open(scores_path, "r", encoding="utf-8") as f:
    scores = json.load(f)

print(f"Test images scored : {scores['n_images_scored']}")
print(f"BLEU-1             : {scores['bleu1']:.4f}")
print(f"BLEU-4             : {scores['bleu4']:.4f}")


### Final measured values

- **Test images scored:** 809
- **BLEU-1:** 0.5008
- **BLEU-4:** 0.1413

BLEU-4 is stricter than BLEU-1, so a lower BLEU-4 is expected when exact multi-word sequences do not match the reference captions.


In [ ]:
pred_path = OUTPUTS / "predictions_test.csv"
predictions = pd.read_csv(pred_path)
print(f"Prediction rows: {len(predictions)}")
display(predictions.head(10))


## 6. Qualitative evaluation

The generated prediction grid contains 10 test images with generated and reference captions. This is the final qualitative evaluation artifact.


In [ ]:
from IPython.display import Image, display

evaluation_image = EVALUATION / "10_test_images.png"
display(Image(filename=str(evaluation_image)))


## 7. Streamlit demo

The completed web demo accepts JPG, JPEG and PNG images and generates a caption using the trained VisionTalk model.

Run it from the project root in PowerShell:

```powershell
streamlit run app\streamlit_app.py
```

The deployed app uses the same VGG16 visual feature extraction and greedy caption-generation approach used for inference.


## 8. Reproducibility commands

For a fresh environment, the original pipeline can be reproduced from the project root with:

```powershell
python src\prepare_captions.py
python src\extract_features.py
python src\train.py
python src\evaluate.py
streamlit run app\streamlit_app.py
```

These commands are intentionally documented here rather than executed automatically in this final-run notebook, because the trained model and evaluation artifacts already exist.


## 9. Final status

### VisionTalk — completed

- [x] Flickr8k dataset prepared
- [x] VGG16 feature extraction completed
- [x] LSTM captioning model trained
- [x] Test-set evaluation completed
- [x] BLEU-1 / BLEU-4 calculated
- [x] Prediction CSV generated
- [x] Qualitative evaluation image generated
- [x] Streamlit demo tested with Flickr8k images

**Final model:** `models/visiontalk_model.keras`

**Final evaluation:** 809 test images, BLEU-1 = 0.5008, BLEU-4 = 0.1413.
